[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/5-memory.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 5. Session and durable memory

The message list is the session. Close it and the sentences are gone. Staff need a guest allergy to outlive that.

This notebook adds `get_preference`. It reads one guest from `data/customer_preference.md`. The model passes a name, not a filename. Staff ask about Priya, then, in a new message list, about Marcus. Each lookup returns that guest's entry. The loop is imported.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, `setup_colab` in `tutorial/common/colab.py` reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `5-memory.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '5-memory'
LESSON_NUMBER = 5
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent


## Tutorial 5 — durable memory

One markdown file holds every guest. A new message list does not delete the file. A name returns one section.

In [ ]:
from tutorial.common.harness import DATA
from tutorial.common.tools import register

PREFERENCE_PATH = DATA / "customer_preference.md"


def _headings(markdown):
    names = []
    for line in markdown.splitlines():
        if line.startswith("## "):
            names.append(line[3:].strip())
    return names


def _section(markdown, heading):
    lines = markdown.splitlines()
    start = None
    prefix = "## " + heading
    for index, line in enumerate(lines):
        if line.strip() == prefix:
            start = index + 1
            break
    if start is None:
        return ""
    body = []
    for line in lines[start:]:
        if line.startswith("## "):
            break
        body.append(line)
    return "\n".join(body).strip()


def guest_list():
    """Comma-separated guest names, in file order."""
    if not PREFERENCE_PATH.is_file():
        return "(none)"
    names = _headings(PREFERENCE_PATH.read_text(encoding="utf-8"))
    if not names:
        return "(none)"
    return ", ".join(names)


def get_preference(args):
    """Return one guest's entry. The model names the guest; this function opens the file."""
    name = str(args.get("name", "")).strip()
    if not name:
        return "ERROR: name is required. Example: Priya."
    if not PREFERENCE_PATH.is_file():
        return "ERROR: customer_preference.md is missing."
    document = PREFERENCE_PATH.read_text(encoding="utf-8")
    match = None
    for heading in _headings(document):
        if heading.lower() == name.lower():
            match = heading
            break
    if match is None:
        return (
            "ERROR: no preference entry for " + name
            + ". Known guests: " + guest_list() + "."
        )
    body = _section(document, match)
    if not body:
        return "ERROR: " + match + " has an empty entry."
    return "CUSTOMER: " + match + "\nSOURCE: data/customer_preference.md\n\n" + body


register(
    "get_preference",
    "Look up one guest in the shared preference file. Pass a name, not a filename. "
    "name is one of: " + guest_list() + ". "
    "The result is that guest's entry only.",
    {
        "name": {
            "type": "string",
            "description": "One guest name: " + guest_list() + ".",
        }
    },
    ["name"],
    get_preference,
)

## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
print("preference file:", PREFERENCE_PATH)
print(PREFERENCE_PATH.read_text(encoding="utf-8"))

print("=== staff ask about Priya; this message list starts empty ===")
priya = run_agent(
    "Priya is at the counter. What allergy do we have on file for her, and what does she usually drink?",
    system_text(),
)
print("ANSWER:", priya["text"])
priya_log = "\n".join(priya["tool_log"])
check(priya["stopped"] == "final", "Priya lookup finished")
check("get_preference" in priya_log, "the preference tool was used")
check("Priya" in priya_log, "the lookup named Priya")
check("Marcus" not in priya_log, "the tool did not dump every guest")
check("almond" in priya["text"].lower(), "the answer names Priya's almond allergy")

print("=== new message list; staff ask about Marcus ===")
marcus = run_agent(
    "Marcus just walked in. Any allergy on file, and what does he usually order?",
    system_text(),
)
print("ANSWER:", marcus["text"])
marcus_log = "\n".join(marcus["tool_log"])
marcus_answer = marcus["text"].lower()
check(marcus["stopped"] == "final", "Marcus lookup finished")
check("get_preference" in marcus_log, "the new session used the preference tool")
check("cardamom" in marcus_answer or "bun" in marcus_answer, "the answer names Marcus's usual order")
check(
    "none" in marcus_answer or "no allergy" in marcus_answer or "not allergic" in marcus_answer,
    "Marcus has no allergy on file",
)